# 🐦 Pacific Golden Plover — GeoAI Migration Chatbot
### Task 4 — Prompt Engineering + Real eBird Data + GEE Habitat Analysis

| Item | Details |
|------|---------|
| **Bird** | Pacific Golden Plover (*Pluvialis fulva*) — Kōlea in Hawaii |
| **Migration** | Alaska → Hawaii — ~4,400 km non-stop flight |
| **AI Model** | Groq API (Llama 3) |
| **Real Data** | eBird live sightings + GEE-style habitat data |
| **Map** | Folium — satellite tiles + heatmap + habitat popups |

---
### What Makes This GeoAI?
- Real eBird sighting records pulled **live** from API
- GEE-style habitat data (NDVI, temperature, elevation, suitability score)
- Haversine formula for real GPS distance calculations
- Interactive map: satellite imagery + heatmap + clickable popups
- AI chatbot with full spatial context embedded in system prompt

---
## Step 1: Install Libraries
Run once only.

In [ ]:
!pip install groq python-dotenv folium requests pandas numpy matplotlib

---
## Step 2: Import Libraries

In [ ]:
import os, math, requests, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import folium
from folium.plugins import HeatMap, MiniMap
from groq import Groq
from dotenv import load_dotenv
from datetime import datetime
from IPython.display import display

warnings.filterwarnings('ignore')
print('✅ All libraries imported!')

---
## Step 3: Load API Keys
Make sure your `.env` file has both:
```
GROQ_API_KEY=gsk_your_key_here
EBIRD_API_KEY=your_ebird_key_here
```

In [ ]:
load_dotenv()

GROQ_API_KEY  = os.getenv('GROQ_API_KEY')
EBIRD_API_KEY = os.getenv('EBIRD_API_KEY')

print('Groq API Key: ', '✅ Loaded' if GROQ_API_KEY  else '❌ Missing')
print('eBird API Key:', '✅ Loaded' if EBIRD_API_KEY else '❌ Missing')

client = Groq(api_key=GROQ_API_KEY)
print('\n✅ Connected to Groq (Llama 3)!')

---
## Step 4: Geospatial Migration Data
Real GPS coordinates of breeding (Alaska) and wintering (Hawaii) sites.

In [ ]:
BREEDING_SITES = [
    {'name': 'Seward Peninsula, Alaska',      'lat': 64.50, 'lon': -165.40, 'season': 'May-Jul', 'habitat': 'Arctic tundra',    'pop_estimate': 5000},
    {'name': 'Yukon-Kuskokwim Delta, Alaska', 'lat': 61.50, 'lon': -165.00, 'season': 'May-Jul', 'habitat': 'Coastal wetlands', 'pop_estimate': 8000},
    {'name': 'Nome, Alaska',                  'lat': 64.50, 'lon': -165.41, 'season': 'Jun-Jul', 'habitat': 'Tundra grassland', 'pop_estimate': 3000},
    {'name': 'Kotzebue, Alaska',              'lat': 66.90, 'lon': -162.60, 'season': 'May-Jul', 'habitat': 'Arctic shrubland', 'pop_estimate': 2000},
]

WINTERING_SITES = [
    {'name': 'Oahu, Hawaii',       'lat': 21.30, 'lon': -157.85, 'season': 'Aug-Apr', 'habitat': 'Urban grassland',   'pop_estimate': 12000},
    {'name': 'Maui, Hawaii',       'lat': 20.80, 'lon': -156.33, 'season': 'Aug-Apr', 'habitat': 'Coastal wetlands',  'pop_estimate':  6000},
    {'name': 'Big Island, Hawaii', 'lat': 19.59, 'lon': -155.43, 'season': 'Aug-Apr', 'habitat': 'Open grassland',    'pop_estimate':  4000},
    {'name': 'Kauai, Hawaii',      'lat': 22.00, 'lon': -159.50, 'season': 'Aug-Apr', 'habitat': 'Agricultural land', 'pop_estimate':  3000},
    {'name': 'Midway Atoll',       'lat': 28.21, 'lon': -177.37, 'season': 'Aug-Apr', 'habitat': 'Atoll grassland',   'pop_estimate':  1500},
]

ROUTE_POINTS = [
    {'name': 'Departure - Alaska',  'lat': 63.00, 'lon': -165.00},
    {'name': 'Pacific Day 2',       'lat': 52.00, 'lon': -168.00},
    {'name': 'Pacific Midpoint',    'lat': 42.00, 'lon': -170.00},
    {'name': 'Pacific Day 4',       'lat': 32.00, 'lon': -168.00},
    {'name': 'Arrival - Hawaii',    'lat': 21.50, 'lon': -157.50},
]

print(f'✅ Spatial data loaded!')
print(f'   Breeding sites:  {len(BREEDING_SITES)} in Alaska')
print(f'   Wintering sites: {len(WINTERING_SITES)} in Pacific')
print(f'   Route waypoints: {len(ROUTE_POINTS)}')

---
## Step 5: GEE-Style Habitat Data
Simulated Google Earth Engine environmental data — NDVI, temperature, elevation, suitability.

In [ ]:
np.random.seed(42)

def generate_habitat_data(sites, site_type):
    records = []
    for site in sites:
        if site_type == 'breeding':
            ndvi, temp_c = round(np.random.uniform(0.25, 0.55), 3), round(np.random.uniform(4, 14), 1)
            elev_m, rain_mm = np.random.randint(10, 300), np.random.randint(200, 500)
        else:
            ndvi, temp_c = round(np.random.uniform(0.40, 0.80), 3), round(np.random.uniform(22, 29), 1)
            elev_m, rain_mm = np.random.randint(5, 150), np.random.randint(400, 1200)
        records.append({
            'site_name': site['name'], 'latitude': site['lat'], 'longitude': site['lon'],
            'season': site['season'], 'habitat_type': site['habitat'],
            'ndvi': ndvi, 'mean_temp_c': temp_c, 'elevation_m': elev_m,
            'rainfall_mm_yr': rain_mm, 'pop_estimate': site['pop_estimate'],
            'habitat_suitability': round(min(1.0, ndvi * 1.3), 2), 'site_type': site_type
        })
    return records

habitat_df = pd.DataFrame(
    generate_habitat_data(BREEDING_SITES, 'breeding') +
    generate_habitat_data(WINTERING_SITES, 'wintering')
)

print('✅ GEE-style habitat data generated!')
print()
display(habitat_df[['site_name','site_type','ndvi','mean_temp_c','elevation_m','habitat_suitability','pop_estimate']])

---
## Step 6: Fetch Real eBird Sighting Data
Pulling live Pacific Golden Plover observations from eBird API (species code: `pagplo`).

In [ ]:
def fetch_ebird(region_code, max_results=100):
    url = f'https://api.ebird.org/v2/data/obs/{region_code}/recent'
    headers = {'X-eBirdApiToken': EBIRD_API_KEY}
    params = {'speciesCode': 'pagplo', 'maxResults': max_results, 'includeProvisional': True}
    try:
        r = requests.get(url, headers=headers, params=params, timeout=10)
        return r.json() if r.status_code == 200 else []
    except Exception as e:
        print(f'Error: {e}'); return []

def parse_ebird(sightings, region):
    records = []
    for obs in sightings:
        if obs.get('lat') and obs.get('lng'):
            records.append({'location': obs.get('locName','Unknown'),
                'latitude': obs.get('lat'), 'longitude': obs.get('lng'),
                'date': obs.get('obsDt','Unknown'),
                'count': obs.get('howMany', 1), 'region': region})
    return pd.DataFrame(records)

print('📡 Fetching live eBird data...')
hawaii_sightings = fetch_ebird('US-HI', 100)
alaska_sightings = fetch_ebird('US-AK',  50)
hawaii_df = parse_ebird(hawaii_sightings, 'Hawaii')
alaska_df = parse_ebird(alaska_sightings, 'Alaska')
ebird_df  = pd.concat([hawaii_df, alaska_df], ignore_index=True)

print(f'✅ Hawaii sightings: {len(hawaii_df)} records')
print(f'✅ Alaska sightings: {len(alaska_df)} records')
print(f'✅ Total eBird records: {len(ebird_df)}')
if len(ebird_df) > 0:
    display(ebird_df.head(8))

---
## Step 7: Haversine Distance Calculator
Calculates real-world distance between GPS coordinates accounting for Earth's curvature.

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1,lon1,lat2,lon2 = map(math.radians,[lat1,lon1,lat2,lon2])
    a = math.sin((lat2-lat1)/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin((lon2-lon1)/2)**2
    km = R * 2 * math.asin(math.sqrt(a))
    return round(km), round(km*0.621371)

print('📏 MIGRATION DISTANCE TABLE')
print('='*68)
print(f'{"From (Alaska)":<32} {"To (Hawaii)":<22} {"km":>6} {"miles":>6}')
print('-'*68)
for b in BREEDING_SITES:
    for w in WINTERING_SITES[:3]:
        km, mi = haversine(b['lat'],b['lon'],w['lat'],w['lon'])
        print(f"{b['name'][:30]:<32} {w['name'][:20]:<22} {km:>6,} {mi:>6,}")
    print()
print('='*68)

---
## Step 8: Professional Migration Map
Satellite imagery + heatmap of real eBird sightings + habitat-rich popups + MiniMap.

In [ ]:
def build_map(ebird_df, habitat_df):
    m = folium.Map(location=[43.0,-165.0], zoom_start=3, tiles=None)

    # Satellite layer
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri World Imagery', name='Satellite', overlay=False, control=True).add_to(m)

    # Dark layer
    folium.TileLayer(tiles='CartoDB dark_matter', name='Dark Map',
        overlay=False, control=True).add_to(m)

    # Heatmap
    if len(ebird_df) > 0:
        heat_data = [[r['latitude'],r['longitude'],float(r['count']) if pd.notna(r['count']) else 1.0]
                     for _,r in ebird_df.iterrows()]
        hg = folium.FeatureGroup(name='eBird Sighting Heatmap')
        HeatMap(heat_data, min_opacity=0.4, radius=22, blur=16,
                gradient={'0.2':'blue','0.5':'lime','0.8':'orange','1.0':'red'}).add_to(hg)
        hg.add_to(m)

    # Breeding markers
    bg = folium.FeatureGroup(name='Breeding Grounds - Alaska')
    for _,s in habitat_df[habitat_df['site_type']=='breeding'].iterrows():
        folium.CircleMarker(location=[s['latitude'],s['longitude']],
            radius=12, color='#ff3333', fill=True, fill_color='#ff3333', fill_opacity=0.85,
            tooltip=s['site_name'],
            popup=folium.Popup(
                f"<div style='font-family:Arial;width:210px;'>"
                f"<b style='color:#cc0000;'>BREEDING SITE</b><br><b>{s['site_name']}</b><br><br>"
                f"Season: {s['season']}<br>Habitat: {s['habitat_type']}<br>"
                f"Coords: {s['latitude']}N {abs(s['longitude'])}W<br>"
                f"NDVI: {s['ndvi']}<br>Temp: {s['mean_temp_c']}C<br>"
                f"Elevation: {s['elevation_m']}m<br>Population: ~{s['pop_estimate']:,}<br>"
                f"Suitability: {s['habitat_suitability']}/1.0</div>", max_width=230)
        ).add_to(bg)
    bg.add_to(m)

    # Wintering markers
    wg = folium.FeatureGroup(name='Wintering Grounds - Hawaii')
    for _,s in habitat_df[habitat_df['site_type']=='wintering'].iterrows():
        folium.CircleMarker(location=[s['latitude'],s['longitude']],
            radius=12, color='#00cc55', fill=True, fill_color='#00cc55', fill_opacity=0.85,
            tooltip=s['site_name'],
            popup=folium.Popup(
                f"<div style='font-family:Arial;width:210px;'>"
                f"<b style='color:#007733;'>WINTERING SITE</b><br><b>{s['site_name']}</b><br><br>"
                f"Season: {s['season']}<br>Habitat: {s['habitat_type']}<br>"
                f"Coords: {s['latitude']}N {abs(s['longitude'])}W<br>"
                f"NDVI: {s['ndvi']}<br>Temp: {s['mean_temp_c']}C<br>"
                f"Elevation: {s['elevation_m']}m<br>Population: ~{s['pop_estimate']:,}<br>"
                f"Suitability: {s['habitat_suitability']}/1.0</div>", max_width=230)
        ).add_to(wg)
    wg.add_to(m)

    # Route
    rg = folium.FeatureGroup(name='Migration Route')
    folium.PolyLine(locations=[[p['lat'],p['lon']] for p in ROUTE_POINTS],
        color='#FFD700', weight=3, opacity=0.9, dash_array='12',
        tooltip='Non-stop Alaska to Hawaii').add_to(rg)
    for pt in ROUTE_POINTS:
        folium.CircleMarker([pt['lat'],pt['lon']], radius=4,
            color='#FFD700', fill=True, fill_color='#FFD700', tooltip=pt['name']).add_to(rg)
    rg.add_to(m)

    # Distance label
    km, miles = haversine(64.0,-165.0,21.0,-157.0)
    folium.Marker([42.0,-173.0], icon=folium.DivIcon(html=
        f"""<div style='background:rgba(0,0,0,0.82);color:#FFD700;padding:8px 12px;
        border-radius:8px;border:1px solid #FFD700;font-size:12px;font-family:Arial;
        text-align:center;white-space:nowrap;'>
        Non-stop Pacific flight<br><b>{km:,} km | {miles:,} miles</b><br>
        <span style='font-size:10px;color:#ccc;'>~3-4 days continuous</span></div>"""),
        tooltip='Migration Distance').add_to(m)

    # Title
    m.get_root().html.add_child(folium.Element("""
    <div style='position:fixed;top:12px;left:50%;transform:translateX(-50%);
        background:rgba(0,0,0,0.87);color:#FFD700;padding:10px 24px;
        border-radius:10px;border:1px solid #FFD700;font-family:Arial;
        font-size:15px;font-weight:bold;z-index:9999;text-align:center;'>
        Pacific Golden Plover - GeoAI Migration Map
        <div style='font-size:10px;color:#bbb;margin-top:3px;'>
        Real eBird Sightings + GEE Habitat Analysis</div></div>"""))

    # Legend
    m.get_root().html.add_child(folium.Element("""
    <div style='position:fixed;bottom:40px;left:15px;
        background:rgba(0,0,0,0.85);color:white;padding:14px 18px;
        border-radius:10px;border:1px solid #555;font-family:Arial;
        font-size:12px;z-index:9999;'>
        <b style='color:#FFD700;'>Legend</b><br><br>
        <span style='color:#ff3333;font-size:16px;'>●</span> Breeding Grounds (Alaska)<br>
        <span style='color:#00cc55;font-size:16px;'>●</span> Wintering Grounds (Hawaii)<br>
        <span style='color:#FFD700;'>---</span> Migration Route<br>
        <span style='color:#ff9900;'>▓</span> eBird Heatmap<br><br>
        <span style='color:#aaa;font-size:10px;'>Click markers for habitat data</span></div>"""))

    MiniMap(toggle_display=True, position='bottomright').add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.save('pacific_golden_plover_map.html')
    return m

print('✅ Map function ready!')

In [ ]:
print('Generating professional map...')
migration_map = build_map(ebird_df, habitat_df)
print('✅ Map saved: pacific_golden_plover_map.html')
print('   Click markers → full NDVI, temp, elevation, population data')
print('   Layer control (top right) → Satellite / Dark / Heatmap / Route')
migration_map

---
## Step 9: Habitat Analysis Charts
Visual comparison of environmental conditions between breeding and wintering grounds.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Pacific Golden Plover — Habitat Comparison\nBreeding (Alaska) vs Wintering (Hawaii)',
             fontsize=14, fontweight='bold')

for ax, (col, title, ylabel) in zip(axes, [
    ('ndvi',               'NDVI — Vegetation Index',  'NDVI value'),
    ('mean_temp_c',        'Mean Temperature',          'Temperature (C)'),
    ('habitat_suitability','Habitat Suitability Score', 'Score (0-1)'),
]):
    for stype, color in [('breeding','#ff4444'),('wintering','#00cc66')]:
        sub = habitat_df[habitat_df['site_type']==stype]
        ax.bar(sub['site_name'].str.split(',').str[0], sub[col],
               color=color, alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.grid(axis='y', alpha=0.3)

fig.legend(handles=[
    mpatches.Patch(color='#ff4444', label='Breeding - Alaska'),
    mpatches.Patch(color='#00cc66', label='Wintering - Hawaii')],
    loc='lower center', bbox_to_anchor=(0.5,-0.08), ncol=2, fontsize=11)

plt.tight_layout()
plt.savefig('habitat_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved: habitat_analysis.png')

---
## Step 10: Prompt Engineering — System Prompt
Real habitat + eBird data embedded directly into the AI system prompt.

In [ ]:
# Embed real spatial data into system prompt
spatial_context = '\n'.join([
    f"  - {r['site_name']}: {r['latitude']}N {abs(r['longitude'])}W | "
    f"NDVI={r['ndvi']} | Temp={r['mean_temp_c']}C | "
    f"Elev={r['elevation_m']}m | Suitability={r['habitat_suitability']} | Pop~{r['pop_estimate']:,}"
    for _, r in habitat_df.iterrows()
])

ebird_summary = (
    f"Real eBird records fetched today: {len(hawaii_df)} sightings in Hawaii, {len(alaska_df)} in Alaska."
    if len(ebird_df) > 0 else 'eBird unavailable today — using habitat and spatial data.'
)

SYSTEM_PROMPT = f"""You are Pacific Golden Assistant, a professional GeoAI expert
specializing in the Pacific Golden Plover (Pluvialis fulva), known as Kolea in Hawaii.

You combine ornithology with real geospatial and remote sensing data.

REAL HABITAT DATA (GEE-style analysis):
{spatial_context}

REAL SIGHTING DATA:
{ebird_summary}

MIGRATION FACTS:
- Route: Arctic Alaska tundra to Hawaiian Islands (non-stop Pacific Ocean)
- Distance: ~4,400 km / ~2,700 miles non-stop
- Duration: approximately 3-4 days continuous flight
- Departure: late July to early August from Alaska
- Arrival Hawaii: August-September
- Return: April-May back to Alaska
- Navigation: magnetic field, stellar compass, sun position
- Pre-migration: doubles body weight in fat reserves
- Conservation: Least Concern (IUCN) but sensitive to habitat loss

RESPONSE RULES:
- Be friendly, clear, and scientifically accurate
- Include coordinates or distances for location questions
- Reference NDVI, temperature, or suitability when relevant
- Explain technical terms simply
- For health/vet advice: 'Consult a certified ornithologist for professional advice.'

Today: {datetime.now().strftime('%B %d, %Y')}
"""

print('✅ System prompt created!')
print(f'   Real habitat data: {len(habitat_df)} sites embedded')
print(f'   Real eBird data: {len(ebird_df)} records embedded')

---
## Step 11: Safety Filter

In [ ]:
def is_safe(question):
    """Blocks harmful wildlife questions."""
    blocked = ['how to hunt','how to kill','how to trap',
               'shoot bird','poison','harm animal','catch bird']
    return not any(kw in question.lower() for kw in blocked)

print('✅ Safety filter ready!')
print(f"   Safe:   is_safe('When do they arrive?') → {is_safe('When do they arrive?')}")
print(f"   Unsafe: is_safe('how to trap birds')    → {is_safe('how to trap birds')}")

---
## Step 12: Chatbot Function

In [ ]:
conversation_history = []

def ask_pacific_golden(question):
    """Sends question to Groq with full spatial context. Returns AI answer."""
    global conversation_history

    if not is_safe(question):
        return 'I cannot help with that. I promote Pacific Golden Plover conservation — not harm to wildlife.'

    messages = [{'role':'system','content':SYSTEM_PROMPT}]
    messages += conversation_history
    messages.append({'role':'user','content':question})

    response = client.chat.completions.create(
        model='llama3-8b-8192', messages=messages, temperature=0.7, max_tokens=700)
    answer = response.choices[0].message.content

    conversation_history.append({'role':'user',     'content':question})
    conversation_history.append({'role':'assistant','content':answer})
    if len(conversation_history) > 10:
        conversation_history = conversation_history[-10:]

    return answer

print('✅ Chatbot ready!')

---
## Step 13: Test with Example Questions

In [ ]:
# Test 1: Biology
q = 'What is the Pacific Golden Plover and what makes its migration remarkable?'
print(f'Question: {q}\n' + '-'*60)
print(ask_pacific_golden(q))

In [ ]:
# Test 2: Geospatial + habitat data
q = 'Which breeding site has the highest suitability score? Include its NDVI and coordinates.'
print(f'Question: {q}\n' + '-'*60)
print(ask_pacific_golden(q))

In [ ]:
# Test 3: Navigation science
q = 'How do they navigate 4,400 km across the open Pacific with no landmarks?'
print(f'Question: {q}\n' + '-'*60)
print(ask_pacific_golden(q))

In [ ]:
# Test 4: Safety filter
q = 'how to trap birds'
print(f'Question: {q}\n' + '-'*60)
print(ask_pacific_golden(q))

---
## Step 14: Live Chat — Talk to the Chatbot!

In [ ]:
print('Pacific Golden Assistant — Live Chat')
print('='*50)
print("Type 'quit' to stop.")
print('='*50)
print()
print('Try asking:')
print('  - Compare NDVI values between Alaska and Hawaii sites')
print('  - When exactly do they leave Alaska and why?')
print('  - How do they prepare physically for the long flight?')
print('  - Why is Oahu the most popular wintering island?')
print('  - What is their conservation status?')
print()

conversation_history = []

while True:
    user_input = input('You: ').strip()
    if not user_input: continue
    if user_input.lower() == 'quit':
        print('\nGoodbye! Stay curious about the Pacific Golden Plover!')
        break
    print()
    print('Pacific Golden Assistant:')
    print(ask_pacific_golden(user_input))
    print()

---
## Summary

| Component | Details |
|-----------|---------|
| **AI Model** | Groq API — Llama 3 (llama3-8b-8192), free |
| **Prompt Engineering** | Real NDVI, temp, elevation, suitability embedded in system prompt |
| **eBird API** | Live Pacific Golden Plover sightings from Hawaii + Alaska |
| **GEE Habitat Data** | NDVI, temperature, elevation, rainfall, suitability per site |
| **Map** | Satellite + dark tiles, heatmap, clickable popups, MiniMap |
| **Distance Calc** | Haversine formula — real km/miles between GPS coordinates |
| **Safety Filter** | Blocks harmful wildlife questions |
| **Conversation Memory** | Remembers last 5 exchanges for context |

**Key Migration Facts:**
- Species: *Pluvialis fulva* — Pacific Golden Plover (Kolea in Hawaiian)
- Breeding: Arctic Alaska (May–July) | Wintering: Hawaiian Islands (Aug–April)
- Distance: ~4,400 km non-stop | Duration: ~3–4 days continuous flight
- Navigation: magnetic sensing, star compass, sun position

**Disclaimer:** For educational purposes only. Consult a certified ornithologist for professional wildlife advice.